# Project 1 — Fairness Audit

**The Price of Fairness: Constrained Risk Pricing**

The baseline prices risk accurately (Week 2–4). Now we ask: *how fair is it?* This notebook measures four families of fairness metrics on the baseline's predictions, then demonstrates why they cannot all hold at once when base rates differ (Chouldechova).

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / "data").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.load import load_claims
from src.models import fit_frequency_model, fit_severity_model
from src.fairness import (
    add_predictions,
    base_rates,
    calibration_by_group,
    demographic_parity,
    equalized_odds,
    premium_shift,
    threshold_for_tpr,
)

plt.rcParams["figure.dpi"] = 110

df = load_claims(ROOT / "data" / "sample_claims.csv")
freq = fit_frequency_model(df)
sev = fit_severity_model(df)
scored = add_predictions(df, freq, sev)
print(f"Scored {len(scored):,} policies")

Scored 10,000 policies


## 1. Base rates — the precondition

If groups were truly identical, fairness would be free. They are not: by construction, gender and territory shift claim frequency and severity. That difference is what makes every fairness metric bite.

In [2]:
base_rates(scored)

n_policies  claim_rate  claim_frequency  \
gender territory                                            
F      A                2039      0.0868           0.0897   
       B                1796      0.0997           0.1091   
       C                1332      0.1456           0.1622   
M      A                1919      0.1058           0.1120   
       B                1674      0.1350           0.1458   
       C                1240      0.1782           0.1992   

                  avg_severity_given_claim  
gender territory                            
F      A                         3024.3550  
       B                         3748.0329  
       C                         5128.7917  
M      A                         3270.3101  
       B                         3871.4389  
       C                         5021.8895

## 2. Demographic parity

*Demographic parity* requires the average score/premium to be the same across groups. The table shows mean predicted premium and frequency per group, relative to the overall mean.

In [3]:
demographic_parity(scored, "predicted_premium")

n_policies  mean_score  ratio_vs_overall  diff_vs_overall
gender territory                                                           
F      A                2039    273.2884            0.5617        -213.2091
       B                1796    392.5270            0.8068         -93.9705
       C                1332    721.7436            1.4836         235.2461
M      A                1919    354.1561            0.7280        -132.3414
       B                1674    491.4007            1.0101           4.9032
       C                1240    918.6845            1.8884         432.1870

In [4]:
demographic_parity(scored, "predicted_frequency")

n_policies  mean_score  ratio_vs_overall  diff_vs_overall
gender territory                                                           
F      A                2039      0.0883            0.6784          -0.0418
       B                1796      0.1130            0.8686          -0.0171
       C                1332      0.1592            1.2239           0.0291
M      A                1919      0.1136            0.8734          -0.0165
       B                1674      0.1416            1.0885           0.0115
       C                1240      0.2023            1.5552           0.0722

## 3. Equalized odds

*Equalized odds* requires true-positive and false-positive rates to be equal across groups at a common decision threshold (here: predicted frequency above the overall mean). With different base rates, this cannot hold alongside calibration — we'll see why below.

In [5]:
equalized_odds(scored, "predicted_frequency", "has_claim")

n     tpr     fpr  predicted_positive_rate
gender territory                                                 
F      A          2039.0  0.2203  0.1278                   0.1359
       B          1796.0  0.4860  0.2900                   0.3096
       C          1332.0  0.7732  0.5492                   0.5818
M      A          1919.0  0.4532  0.2890                   0.3064
       B          1674.0  0.6549  0.4496                   0.4773
       C          1240.0  0.9140  0.7664                   0.7927

## 4. Calibration parity

*Calibration* requires the same predicted score to mean the same risk in every group: average predicted premium should equal average actual loss within each group. This is the property the baseline *does* satisfy.

In [6]:
calibration_by_group(scored, "total_claim_amount", "predicted_premium")

n_policies  actual_mean  predicted_mean  predicted/actual
gender territory                                                           
F      A                2039     262.5360        273.2884            1.0410
       B                1796     373.5512        392.5270            1.0508
       C                1332     746.9862        721.7436            0.9662
M      A                1919     345.9473        354.1561            1.0237
       B                1674     522.6674        491.4007            0.9402
       C                1240     895.0303        918.6845            1.0264

In [7]:
calibration_by_group(scored, "claim_count", "predicted_frequency")

n_policies  actual_mean  predicted_mean  predicted/actual
gender territory                                                           
F      A                2039       0.0897          0.0883            0.9833
       B                1796       0.1091          0.1130            1.0354
       C                1332       0.1622          0.1592            0.9820
M      A                1919       0.1120          0.1136            1.0142
       B                1674       0.1458          0.1416            0.9715
       C                1240       0.1992          0.2023            1.0158

## 5. The impossibility, demonstrated

Chouldechova's result: when base rates differ, you cannot satisfy equalized odds **and** calibration at the same time. Concretely — to give every group the same true-positive rate (0.5 here), each group needs its own threshold. The thresholds differ:

* F would need threshold ≈ 0.120
* M would need threshold ≈ 0.150

A single score means different risk in the two groups (calibration) *or* the groups are treated differently (equalized odds). Pick one.

In [8]:
threshold_for_tpr(scored, "predicted_frequency", "has_claim", target_tpr=0.5)

needed_threshold  base_rate
gender territory                             
F      A                    0.0964     0.0868
       B                    0.1289     0.0997
       C                    0.1762     0.1456
M      A                    0.1196     0.1058
       B                    0.1542     0.1350
       C                    0.2076     0.1782

## 6. Premium shift — who pays more?

The baseline premium spread is the raw material for the "price of fairness": the most expensive segment pays roughly twice the cheapest. Constrained pricing will compress this spread and show what that costs.

In [9]:
premium_shift(scored, "predicted_premium")

,,mean_premium,premium_ratio_vs_cheapest,premium_gap_vs_cheapest
gender,territory,,,
F,A,273.29,1.00,0.00
M,A,354.16,1.30,80.87
F,B,392.53,1.44,119.24
M,B,491.40,1.80,218.11
F,C,721.74,2.64,448.46
M,C,918.68,3.36,645.40


## Observations

* **Calibration holds**: predicted premiums match actual losses within ~2% in every group — the model is accurate.
* **Demographic parity fails**: M pays 12% above the overall average premium, F pays 11% below.
* **Equalized odds fails**: TPR is 0.68 for M vs 0.42 for F at a common threshold — the model flags male claims more often.
* **The conflict is structural, not a bug**: with different base rates, fixing equalized odds would break calibration, and fixing demographic parity would break both. This is why fairness is a *choice* — and the next milestone (constrained pricing) will quantify the cost of each choice with the accuracy-fairness frontier.